# Flipkart Gridlock 2.0 — Version 3 Pipeline
## Unified Spatiotemporal Architecture

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import pygeohash as pgh
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
print("[INGESTION] Locating and streaming raw datasets...")

POSSIBLE_ROUTES = [".", "data/raw", "../../data/raw"]
data_path = next((r for r in POSSIBLE_ROUTES if os.path.exists(os.path.join(r, "train.csv"))), None)

if not data_path:
    raise FileNotFoundError("Raw source CSV files missing.")

raw_train = pd.read_csv(os.path.join(data_path, "train.csv"))
raw_test = pd.read_csv(os.path.join(data_path, "test.csv"))

y_target = raw_train['demand']
test_submission_index = raw_test['Index'].values

X_train_base = raw_train.drop(columns=['demand'], errors='ignore')
X_test_base = raw_test.copy()

[INGESTION] Locating and streaming raw datasets...


In [3]:
print("[ENGINEERING] Feature 1: Generating exact 24-hour demand lags...")

lag_lookup = raw_train[['geohash', 'day', 'timestamp', 'demand']].copy()
lag_lookup['day'] = lag_lookup['day'] + 1
lag_lookup.rename(columns={'demand': 'lag_demand_24h'}, inplace=True)

X_train_lag = pd.merge(X_train_base, lag_lookup, on=['geohash', 'day', 'timestamp'], how='left')
X_test_lag = pd.merge(X_test_base, lag_lookup, on=['geohash', 'day', 'timestamp'], how='left')

global_mean = y_target.mean()
X_train_lag['lag_demand_24h'] = X_train_lag['lag_demand_24h'].fillna(global_mean)
X_test_lag['lag_demand_24h'] = X_test_lag['lag_demand_24h'].fillna(global_mean)

[ENGINEERING] Feature 1: Generating exact 24-hour demand lags...


In [4]:
print("[ENGINEERING] Features 2 & 3: Cyclic time waves and Spatial Interactions...")

def parse_time(ts_string):
    h, m = ts_string.split(":")
    return int(h) * 60 + int(m)

def decode_geo(gh_string):
    try:
        lat, lon = pgh.decode(gh_string)
        return lat, lon
    except Exception:
        return np.nan, np.nan

def synthesize_core_features(df, freq_maps=None):
    df_f = df.copy()
    
    df_f['Temperature'] = df_f['Temperature'].fillna(df_f['Temperature'].median())
    df_f['Weather'] = df_f['Weather'].fillna('Unknown')
    df_f['RoadType'] = df_f['RoadType'].fillna('Unknown')
    
    df_f['ts_minutes'] = df_f['timestamp'].apply(parse_time)
    df_f['hour'] = df_f['ts_minutes'] // 60
    df_f['minute'] = df_f['ts_minutes'] % 60
    df_f['time_slot_15m'] = df_f['ts_minutes'] // 15
    
    # Feature 2: Time Series Cyclic Transformation
    df_f['hour_sin'] = np.sin(2 * np.pi * df_f['hour'] / 24.0)
    df_f['hour_cos'] = np.cos(2 * np.pi * df_f['hour'] / 24.0)
    df_f['min_sin'] = np.sin(2 * np.pi * df_f['ts_minutes'] / 1440.0)
    df_f['min_cos'] = np.cos(2 * np.pi * df_f['ts_minutes'] / 1440.0)
    
    df_f['is_peak_am'] = df_f['hour'].between(7, 9).astype(int)
    df_f['is_peak_pm'] = df_f['hour'].between(17, 19).astype(int)
    df_f['is_night'] = (df_f['hour'] < 6).astype(int)
    df_f['is_midnight'] = (df_f['hour'] == 0).astype(int)
    df_f['is_weekend'] = (df_f['day'] % 7).isin([5, 6]).astype(int)
    df_f['is_rush_hour'] = df_f['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    coords = df_f['geohash'].apply(decode_geo)
    df_f['lat'] = coords.apply(lambda x: x[0])
    df_f['lon'] = coords.apply(lambda x: x[1])
    df_f['geohash_prefix3'] = df_f['geohash'].str[:3]
    df_f['geohash_prefix4'] = df_f['geohash'].str[:4]
    
    df_f['temp_sq'] = df_f['Temperature'] ** 2
    df_f['peak_x_lanes'] = (df_f['is_peak_am'] + df_f['is_peak_pm']) * df_f['NumberofLanes']
    df_f['lat_x_lon'] = df_f['lat'] * df_f['lon']
    
    v_weight = df_f['LargeVehicles'].map({'Allowed': 1.0, 'Not Allowed': 0.5}).fillna(0.75)
    df_f['Traffic_Capacity_Index'] = df_f['NumberofLanes'] * v_weight
    
    if freq_maps is None:
        freq_maps = {
            'geo_freq': df_f['geohash'].value_counts().to_dict(),
            'time_freq': df_f['time_slot_15m'].value_counts().to_dict()
        }
    
    df_f['geo_density'] = df_f['geohash'].map(freq_maps['geo_freq']).fillna(1)
    df_f['time_density'] = df_f['time_slot_15m'].map(freq_maps['time_freq']).fillna(1)
    
    # Feature 3: Geohash x Timeslot Interaction
    df_f['interaction_geo_time'] = df_f['geohash'].astype(str) + "_" + df_f['time_slot_15m'].astype(str)
    
    return df_f, freq_maps

X_train_fe, train_freq_maps = synthesize_core_features(X_train_lag, freq_maps=None)
X_test_fe, _ = synthesize_core_features(X_test_lag, freq_maps=train_freq_maps)

[ENGINEERING] Features 2 & 3: Cyclic time waves and Spatial Interactions...


In [5]:
print("[ENGINEERING] Unsupervised K-Means Spatial Clustering...")

global_coords = pd.concat([X_train_fe[['lat', 'lon']], X_test_fe[['lat', 'lon']]]).dropna()
kmeans_model = KMeans(n_clusters=25, random_state=42, n_init=10).fit(global_coords)

X_train_fe['spatial_hub_cluster'] = kmeans_model.predict(X_train_fe[['lat', 'lon']].fillna(0))
X_test_fe['spatial_hub_cluster'] = kmeans_model.predict(X_test_fe[['lat', 'lon']].fillna(0))

center_lat, center_lon = global_coords['lat'].mean(), global_coords['lon'].mean()
X_train_fe['distance_to_center'] = np.sqrt((X_train_fe['lat'] - center_lat)**2 + (X_train_fe['lon'] - center_lon)**2)
X_test_fe['distance_to_center'] = np.sqrt((X_test_fe['lat'] - center_lat)**2 + (X_test_fe['lon'] - center_lon)**2)

[ENGINEERING] Unsupervised K-Means Spatial Clustering...


In [6]:
print("[ENGINEERING] Feature 4: Leakage-Free K-Fold Target Profiling...")

def get_kfold_target_encoding(train_df, test_df, target_series, col, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    train_encoded = np.zeros(len(train_df))
    
    w_train, w_test = train_df.copy(), test_df.copy()
    w_train["_tgt_"] = target_series.values
    
    global_mean = target_series.mean()
    global_map = w_train.groupby(col)["_tgt_"].mean()
    test_encoded = w_test[col].map(global_map).fillna(global_mean).values
    
    for tr_idx, val_idx in kf.split(w_train):
        fold_map = w_train.iloc[tr_idx].groupby(col)["_tgt_"].mean()
        train_encoded[val_idx] = w_train.iloc[val_idx][col].map(fold_map).fillna(global_mean).values
        
    return train_encoded, test_encoded

X_train_fe['gh_demand_mean'], X_test_fe['gh_demand_mean'] = get_kfold_target_encoding(
    X_train_fe, X_test_fe, y_target, 'geohash'
)

X_train_fe['TE_geo_time'], X_test_fe['TE_geo_time'] = get_kfold_target_encoding(
    X_train_fe, X_test_fe, y_target, 'interaction_geo_time'
)

gh_stats = raw_train.groupby("geohash")["demand"].agg(gh_demand_std="std", gh_demand_max="max", gh_demand_median="median").reset_index()
gh_stats["gh_demand_std"] = gh_stats["gh_demand_std"].fillna(0)

X_train_fe = X_train_fe.merge(gh_stats, on="geohash", how="left")
X_test_fe = X_test_fe.merge(gh_stats, on="geohash", how="left")

for c in ["gh_demand_std", "gh_demand_max", "gh_demand_median"]:
    X_train_fe[c] = X_train_fe[c].fillna(X_train_fe[c].mean())
    X_test_fe[c] = X_test_fe[c].fillna(X_train_fe[c].mean())

[ENGINEERING] Feature 4: Leakage-Free K-Fold Target Profiling...


In [7]:
print("[PROCESS] Categorical Encoding & Schema Locking...")

cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_prefix3', 'geohash_prefix4']

road_map = {"Highway": 3, "Street": 2, "Residential": 1, "Unknown": 0}
weather_map = {"Sunny": 4, "Cloudy": 3, "Rainy": 2, "Foggy": 1, "Snowy": 0, "Unknown": -1}
X_train_fe['RoadType_enc'] = X_train_fe['RoadType'].map(road_map).fillna(0).astype(int)
X_test_fe['RoadType_enc'] = X_test_fe['RoadType'].map(road_map).fillna(0).astype(int)
X_train_fe['Weather_enc'] = X_train_fe['Weather'].map(weather_map).fillna(-1).astype(int)
X_test_fe['Weather_enc'] = X_test_fe['Weather'].map(weather_map).fillna(-1).astype(int)

X_train_fe.drop(columns=['timestamp', 'interaction_geo_time', 'Index'], errors='ignore', inplace=True)
X_test_fe.drop(columns=['timestamp', 'interaction_geo_time', 'Index'], errors='ignore', inplace=True)

for c in cat_cols:
    le = LabelEncoder()
    le.fit(X_train_fe[c].astype(str).tolist() + X_test_fe[c].astype(str).tolist())
    X_train_fe[c] = le.transform(X_train_fe[c].astype(str))
    X_test_fe[c] = le.transform(X_test_fe[c].astype(str))

X_train_fe['geohash_enc'] = X_train_fe['geohash']
X_test_fe['geohash_enc'] = X_test_fe['geohash']
X_train_fe['LargeVehicles_enc'] = X_train_fe['LargeVehicles']
X_test_fe['LargeVehicles_enc'] = X_test_fe['LargeVehicles']
X_train_fe['Landmarks_enc'] = X_train_fe['Landmarks']
X_test_fe['Landmarks_enc'] = X_test_fe['Landmarks']
X_train_fe['temp_x_weather'] = X_train_fe['Temperature'] * X_train_fe['Weather_enc']
X_test_fe['temp_x_weather'] = X_test_fe['Temperature'] * X_test_fe['Weather_enc']

# Exactly 37 Features
FEATURE_COLS = [
    "day", "ts_minutes", "hour", "minute", "is_peak_am", "is_peak_pm", "is_night", "is_midnight",
    "is_weekend", "is_rush_hour", "hour_sin", "hour_cos", "min_sin", "min_cos", "lat", "lon", "lat_x_lon",
    "geohash_prefix3", "geohash_prefix4", "geohash_enc", "NumberofLanes",
    "RoadType_enc", "Weather_enc", "LargeVehicles_enc", "Landmarks_enc",
    "Temperature", "temp_sq", "temp_x_weather", "peak_x_lanes", "lag_demand_24h",
    "Traffic_Capacity_Index", "geo_density", "time_density",
    "spatial_hub_cluster", "distance_to_center",
    "gh_demand_mean", "gh_demand_std", "gh_demand_max", "gh_demand_median", "TE_geo_time"
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in X_train_fe.columns]

X = X_train_fe[FEATURE_COLS].values
y = y_target.values
X_test = X_test_fe[FEATURE_COLS].values
print(f"[STATUS] Matrix Locked. Train: {X.shape} | Test: {X_test.shape}")

[PROCESS] Categorical Encoding & Schema Locking...
[STATUS] Matrix Locked. Train: (77299, 40) | Test: (41778, 40)


In [8]:
print("[TRAINING] Multi-Engine Cross-Validation Loop...")

NUM_FOLDS = 5
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

oof_lgb, oof_xgb, oof_cat = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))
test_lgb, test_xgb, test_cat = np.zeros(len(X_test)), np.zeros(len(X_test)), np.zeros(len(X_test))

lgb_params = {
    'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.03, 'num_leaves': 256, 
    'max_depth': -1, 'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 
    'bagging_freq': 5, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'verbose': -1, 'random_state': 42, 'n_jobs': -1
}
xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03, 'max_depth': 7, 
    'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 
    'min_child_weight': 5, 'random_state': 42, 'n_jobs': -1
}
cat_params = {
    'iterations': 2000, 'learning_rate': 0.03, 'depth': 8, 'l2_leaf_reg': 3, 'subsample': 0.8, 
    'colsample_bylevel': 0.8, 'eval_metric': 'RMSE', 'early_stopping_rounds': 100, 
    'random_seed': 42, 'verbose': 0, 'allow_writing_files': False
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    
    model_lgb = lgb.train(lgb_params, lgb.Dataset(X_tr, label=y_tr), num_boost_round=2500, valid_sets=[lgb.Dataset(X_va, label=y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    test_lgb += model_lgb.predict(X_test) / NUM_FOLDS
    
    model_xgb = xgb.train(xgb_params, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=2500, evals=[(xgb.DMatrix(X_va, label=y_va), 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[val_idx] = model_xgb.predict(xgb.DMatrix(X_va))
    test_xgb += model_xgb.predict(xgb.DMatrix(X_test)) / NUM_FOLDS
    
    model_cat = CatBoostRegressor(**cat_params).fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[val_idx] = model_cat.predict(X_va)
    test_cat += model_cat.predict(X_test) / NUM_FOLDS
    
    print(f"  -> Fold {fold} LightGBM Base Metric: {max(0, 100 * r2_score(y_va, oof_lgb[val_idx])):.4f}")

[TRAINING] Multi-Engine Cross-Validation Loop...
  -> Fold 1 LightGBM Base Metric: 96.0812
  -> Fold 2 LightGBM Base Metric: 96.0388
  -> Fold 3 LightGBM Base Metric: 96.1338
  -> Fold 4 LightGBM Base Metric: 95.7616
  -> Fold 5 LightGBM Base Metric: 96.0839


In [10]:
print("[OPTIMIZE] Direct Metric Optimization & Export...")

def target_metric_objective(weights):
    w = np.array(weights)
    if w.sum() == 0: return 999.0
    w = w / w.sum()
    
    blend = (w[0] * oof_lgb) + (w[1] * oof_xgb) + (w[2] * oof_cat)
    score = max(0, 100 * r2_score(y, blend))
    return -score # Minimize negative score to maximize target metric

solver = minimize(target_metric_objective, [0.33, 0.33, 0.33], method='Nelder-Mead')
w_opt = solver.x / sum(solver.x)

final_r2 = r2_score(y, (w_opt[0]*oof_lgb + w_opt[1]*oof_xgb + w_opt[2]*oof_cat))
final_score = max(0, 100 * final_r2)

print(f"\n Optimized Blend Ratios -> LGBM: {w_opt[0]:.3f} | XGB: {w_opt[1]:.3f} | CAT: {w_opt[2]:.3f}")
print(f" Expected Leaderboard Score: {final_score:.4f}\n")

# Generate final clipped predictions
final_preds = np.clip((w_opt[0]*test_lgb + w_opt[1]*test_xgb + w_opt[2]*test_cat), 0.0, 1.0)

# Build schema matching the 41778 length submission file exactly
submission_df = pd.DataFrame({'Index': test_submission_index, 'demand': final_preds})
submission_df.to_csv("submission_3.csv", index=False)

print(f" SUBMISSION ROWS    : {submission_df.shape[0]} (Target: 41778)")
print(f" SUBMISSION COLUMNS : {list(submission_df.columns)} (Target: ['Index', 'demand'])")
print("File saved as 'submission_3.csv'. Ready for upload!")

[OPTIMIZE] Direct Metric Optimization & Export...

 Optimized Blend Ratios -> LGBM: 0.309 | XGB: 0.426 | CAT: 0.265
 Expected Leaderboard Score: 96.1508

 SUBMISSION ROWS    : 41778 (Target: 41778)
 SUBMISSION COLUMNS : ['Index', 'demand'] (Target: ['Index', 'demand'])
File saved as 'submission_3.csv'. Ready for upload!
